In [ ]:
'''
pip install langchain
pip install dotenv
pip install langchain-community
pip install langchain-google-genai
pip install google-search-results
pip install bs4
pip install requests
'''

In [115]:
from langchain.agents import AgentExecutor, create_tool_calling_agent, Tool
from langchain_core.prompts import ChatPromptTemplate
from langchain.utilities import SerpAPIWrapper
from langchain_core.messages import AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from serpapi import GoogleSearch
import re
import os
from dotenv import load_dotenv

In [116]:
load_dotenv("secrets.env")
my_key = os.getenv("GEMINI_KEY")
serp_key = os.getenv("SERPAPI_API_KEY")

os.environ["GOOGLE_API_KEY"] = my_key
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-001", temperature=0.7)

In [127]:
EVENT_SITES = [
    "https://www.eventbrite.com/d/ny--new-york/"
]

In [128]:
def event_search(query: str, max_results: int = 10) -> str:
    filter_domains = " OR ".join([f"site:{domain}" for domain in EVENT_SITES])
    full_query = f"{query} {filter_domains}"

    search = SerpAPIWrapper()

    try:
        raw_results = search.results(full_query)

        results_text = []
        count = 0

        for result in raw_results.get("organic_results", []):
            url = result.get("link", "")
            if any(domain in url for domain in EVENT_SITES):
                title = result.get("title", "")
                snippet = result.get("snippet", "")
                results_text.append(f"Title: {title}\nSnippet: {snippet}\nSource: {url}")
                count += 1
            if count >= max_results:
                break

        return "\n\n".join(results_text) if results_text else "No events found."

    except Exception as e:
        return f"Search failed: {str(e)}"

In [129]:
tools = [
    Tool(
        name="EventSearch",
        func=event_search,
        description="Searches relevant events from Eventbrite"
    )
]

In [130]:
prompt = ChatPromptTemplate.from_messages(
    [("system", (
        "You are an events planner and you have access to a specialized web tool"
        "Given the user's inputs of location and profile, match events that they will find enjoyable"
        "For the user to enjoy the event they need the event to match ONE or more of their interests"
        "Not all interests need to match the event"
        "Use the tool to search for these events"
        )),
        ("user", "{input}"),
        AIMessage(content="Okay, I will use the trusted search tool if necessary to find the answer from online sources."),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

In [131]:
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True, return_intermediate_steps=True)

In [132]:
interests = 'art'
date = '10/04'

In [133]:
user_input = f"What {interests} events are happening this week"

In [134]:
response = agent_executor.invoke({"input": user_input})
print(f"{response['output']}") 



> Entering new AgentExecutor chain...

Invoking: `EventSearch` with `art events this week`
responded: 


Title: Best Arts Events This Week in New York, NY
Snippet: Looking for arts events this week in New York, NY? Explore your interests with all sorts of unique things to do near you.
Source: https://www.eventbrite.com/d/ny--new-york/arts--events--this-week/

Title: Free New York, NY Face Painting And Body Art ...
Snippet: Free face painting and body art convention las vegas events in New York, NY · Today · Tomorrow · This weekend · This week · Next week · This month · Next month.
Source: https://www.eventbrite.com/d/ny--new-york/free--events/face-painting-and-body-art-convention-las-vegas/

Title: Best Arts Events This Weekend in New York, NY
Snippet: Looking for arts events this weekend in New York, NY? Explore your interests with all sorts of unique things to do near you.
Source: https://www.eventbrite.com/d/ny--new-york/arts--events--this-weekend/

Title: Discover Art Events & Ac

In [135]:
if 'intermediate_steps' in response and response['intermediate_steps']:
    print("\n--- Agent's Intermediate Steps (Context & Raw Tool Output) ---")
    for step_number, (action, observation) in enumerate(response['intermediate_steps']):
        print(f"Step {step_number + 1}:")
        print(f"  Action Tool: {action.tool}")
        print(f"  Action Input: {action.tool_input}")
        print(f"  Observation (Raw Tool Output):\n{observation}\n---")


--- Agent's Intermediate Steps (Context & Raw Tool Output) ---
Step 1:
  Action Tool: EventSearch
  Action Input: art events this week
  Observation (Raw Tool Output):
Title: Best Arts Events This Week in New York, NY
Snippet: Looking for arts events this week in New York, NY? Explore your interests with all sorts of unique things to do near you.
Source: https://www.eventbrite.com/d/ny--new-york/arts--events--this-week/

Title: Free New York, NY Face Painting And Body Art ...
Snippet: Free face painting and body art convention las vegas events in New York, NY · Today · Tomorrow · This weekend · This week · Next week · This month · Next month.
Source: https://www.eventbrite.com/d/ny--new-york/free--events/face-painting-and-body-art-convention-las-vegas/

Title: Best Arts Events This Weekend in New York, NY
Snippet: Looking for arts events this weekend in New York, NY? Explore your interests with all sorts of unique things to do near you.
Source: https://www.eventbrite.com/d/ny--new-yor

In [85]:
from typing import List, Dict

In [ ]:
serp_key = os.getenv("SERPAPI_API_KEY")

In [107]:
def search_eventbrite(query: str, location: str, max_results: int = 10) -> List[Dict]:
  
    search_query = f'site:eventbrite.com "{query}" "{location}"'
    
    params = {
        "engine": "google",
        "q": search_query,
        "api_key": serp_key,
        "num": max_results
    }
    
    search = GoogleSearch(params)
    results = search.get_dict()
    
    events = []
    for r in results.get("organic_results", [])[:max_results]:
        events.append({
            "title": r.get("title"),
            "url": r.get("link"),
            "snippet": r.get("snippet"),
            "source": "eventbrite"
        })
    return events

In [ ]:
import re

def is_actual_event(url: str) -> bool:

    return re.search(r"-tickets-\d+$", url) is not None

In [142]:
results = search_eventbrite("data science", "New York", max_results=10)

In [143]:
for event in results:
    if is_actual_event(event['url']):
        print(event)

{'title': 'Data Science & Analytics meetup and chat New York Friday', 'url': 'https://www.eventbrite.com/e/data-science-analytics-meetup-and-chat-new-york-tickets-1059285885599', 'snippet': 'Eventbrite - IT Social New York presents Data Science & Analytics meetup and chat New York Friday - Friday, 1 November 2024 | Friday, 19 December 2025 at ...', 'source': 'eventbrite'}
{'title': 'Data Science Day 2025', 'url': 'https://www.eventbrite.com/e/data-science-day-2025-tickets-1156359555559', 'snippet': 'Eventbrite - Data Science Institute, Columbia University presents Data Science Day 2025 - Wednesday, April 2, 2025 at Alfred Lerner Hall, New York, NY.', 'source': 'eventbrite'}
{'title': 'Fall 2025 Lecture in Climate Data Science: HAMIDREZA ...', 'url': 'https://www.eventbrite.com/e/fall-2025-lecture-in-climate-data-science-hamidreza-norouzi-tickets-1607254083229', 'snippet': 'Eventbrite - LEAP Center presents Fall 2025 Lecture in Climate Data Science: HAMIDREZA NOROUZI - Thursday, Decembe

In [ ]:
import requests
from bs4 import BeautifulSoup
import json

In [152]:
listing_url = "https://www.eventbrite.com/d/ny--new-york/arts--events--this-week/"
r = requests.get(listing_url, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(r.text, "html.parser")

event_links = []
for a in soup.find_all("a", href=True):
    href = a['href']
    if is_actual_event(href):
        event_links.append(href)

In [153]:
event_links

['https://www.eventbrite.com/e/jawani-nycs-biggest-panjabi-boiler-room-rave-ladies-free-tickets-1559521222959',
 'https://www.eventbrite.com/e/it-s-a-fake-wedding-nyc-tickets-1570462207739',
 'https://www.eventbrite.com/e/chronicles-of-trevor-tickets-1496119456569',
 'https://www.eventbrite.com/e/kids-teach-kids-listen-child-made-movie-premiere-tickets-1629233253519',
 'https://www.eventbrite.com/e/bcco-fall-outdoor-concert-at-the-old-stone-house-tickets-1339995987069',
 'https://www.eventbrite.com/e/deviant-acts-standup-from-a-media-pro-turned-media-prankster-tickets-1364156261099',
 'https://www.eventbrite.com/e/standup-comedy-at-concrete-shoals-tickets-1689077539389',
 'https://www.eventbrite.com/e/thirteen-tickets-1730512021039']

In [ ]:
url = "https://www.eventbrite.com/e/thirteen-tickets-1730512021039"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
})

response = session.get(url)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")


meta_tags = {meta.get("property") or meta.get("name"): meta.get("content")
             for meta in soup.find_all("meta") if meta.get("content")}

print("Meta tags found:", len(meta_tags))


Meta tags found: 43


In [165]:
keys_to_keep = [
    'og:title',
    'og:description',
    'og:url',
    'event:start_time',
    'event:end_time',
    'event:location:latitude',
    'event:location:longitude',  
    'og:image'
]

filtered_meta_tags = {k: v for k, v in meta_tags.items() if k in keys_to_keep}
filtered_meta_tags

{'og:image': 'https://cdn.evbuc.com/images/1131387233/448023183754/1/logo.20250922-141630',
 'og:title': 'THIRTEEN',
 'og:description': 'Fun, silly, low stakes. But still a full body workout. And a full hearted time. A dance class that doesn’t take itself seriously.',
 'og:url': 'https://www.eventbrite.com/e/thirteen-tickets-1730512021039',
 'event:location:latitude': '40.6869833',
 'event:location:longitude': '-73.97891680000001',
 'event:start_time': '2025-09-26T19:15:00-04:00',
 'event:end_time': '2025-09-26T20:45:00-04:00'}

In [168]:
match = re.search(r'-tickets-(\d+)', filtered_meta_tags['og:url'])
if match:
    filtered_meta_tags['id'] = match.group(1)

filtered_meta_tags

{'og:image': 'https://cdn.evbuc.com/images/1131387233/448023183754/1/logo.20250922-141630',
 'og:title': 'THIRTEEN',
 'og:description': 'Fun, silly, low stakes. But still a full body workout. And a full hearted time. A dance class that doesn’t take itself seriously.',
 'og:url': 'https://www.eventbrite.com/e/thirteen-tickets-1730512021039',
 'event:location:latitude': '40.6869833',
 'event:location:longitude': '-73.97891680000001',
 'event:start_time': '2025-09-26T19:15:00-04:00',
 'event:end_time': '2025-09-26T20:45:00-04:00',
 'id': '1730512021039'}

In [161]:
with open("event_meta.json", "w", encoding="utf-8") as f:
    import json
    json.dump(meta_tags, f, indent=2, ensure_ascii=False)